# Clase 6 — Práctica: el problema XOR

Esta práctica retoma el circuito de la clase de teoría y te propone quitarle una pieza para comprobar, con tus propios ojos, por qué esa pieza es imprescindible.

## Ponlo en práctica: elimina el `CNOT` inicial

En la clase de teoría viste que el circuito empieza con `circuito.append(cirq.CNOT(qubits[0], qubits[1]))`, una puerta que entrelaza los dos qubits de entrada antes de aplicar ninguna rotación entrenable. La explicación teórica dice que ese `CNOT` inicial es la fuente de la no linealidad que permite resolver el XOR.

Tu tarea es comprobarlo de forma experimental:

1. Copia `construir_circuito` y elimina de ella la línea del `CNOT` inicial (deja el resto exactamente igual)
2. Reentrena el modelo desde cero con las mismas 512 épocas y los mismos datos del XOR
3. Vuelve a dibujar la frontera de decisión con el nuevo modelo
4. Compara la precisión final (`precision_hinge`) del modelo sin `CNOT` inicial con la del modelo original
5. Anota qué diferencia observas y relaciónala con el papel del entrelazamiento: ¿el circuito sin `CNOT` inicial se comporta como dos clasificadores independientes, igual que un perceptrón clásico?

**Pistas**

- Reutiliza `codificar_dato`, `generar_datos_xor`, `rotacion_de_un_qubit`, `agrupamiento_dos_qubits` y `precision_hinge` tal cual, no cambian
- Solo tienes que tocar el cuerpo de `construir_circuito`
- Usa un nombre de variable distinto para el nuevo modelo (por ejemplo `modelo_sin_entrelazamiento`) para no perder el modelo original y poder comparar ambos

In [ ]:
import tensorflow_quantum as tfq
import tensorflow as tf
import cirq
import sympy
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap


def codificar_dato(qubits, circuito, dato):
    """X^i = Rx(pi * i): permite codificar valores continuos entre 0 y 1."""
    circuito.append([cirq.X(qubits[0]) ** dato[0]])
    circuito.append([cirq.X(qubits[1]) ** dato[1]])
    return circuito


def generar_datos_xor(qubits):
    datos = [[0, 0], [1, 0], [0, 1], [1, 1]]
    etiquetas = [-1, 1, 1, -1]
    circuitos = [codificar_dato(qubits, cirq.Circuit(), d) for d in datos]
    return tfq.convert_to_tensor(circuitos), np.array(etiquetas)


def rotacion_de_un_qubit(qubit, simbolos):
    return cirq.Circuit(
        cirq.X(qubit) ** simbolos[0],
        cirq.Y(qubit) ** simbolos[1],
        cirq.Z(qubit) ** simbolos[2],
    )


def agrupamiento_dos_qubits(fuente, receptor, simbolos):
    """Combina la información de dos qubits en la respuesta de uno solo (el receptor)."""
    circuito = cirq.Circuit()
    selector_receptor = rotacion_de_un_qubit(receptor, simbolos[0:3])
    selector_fuente = rotacion_de_un_qubit(fuente, simbolos[3:6])
    circuito.append(selector_receptor)
    circuito.append(selector_fuente)
    circuito.append(cirq.CNOT(control=fuente, target=receptor))
    circuito.append(selector_receptor ** -1)
    return circuito


def precision_hinge(etiqueta_real, prediccion):
    etiqueta_real = tf.squeeze(etiqueta_real) > 0.0
    prediccion = tf.squeeze(prediccion) > 0.0
    return tf.reduce_mean(tf.cast(etiqueta_real == prediccion, tf.float32))


qubits = [cirq.GridQubit(0, i) for i in range(2)]
entrenamiento, etiquetas = generar_datos_xor(qubits)

In [ ]:
def construir_circuito_sin_entrelazamiento(qubits):
    """Igual que construir_circuito, pero sin el CNOT inicial que entrelaza los qubits de entrada."""
    x1, y1, z1, x2, y2, z2 = sympy.symbols("x1 y1 z1 x2 y2 z2")
    agrupamiento = sympy.symbols("agrupamiento0:6")
    circuito = cirq.Circuit()
    # TODO: no añadas aquí el CNOT inicial entre qubits[0] y qubits[1]
    circuito.append(cirq.rx(x1).on(qubits[0]))
    circuito.append(cirq.ry(y1).on(qubits[0]))
    circuito.append(cirq.rz(z1).on(qubits[0]))
    circuito.append(cirq.rx(x2).on(qubits[1]))
    circuito.append(cirq.ry(y2).on(qubits[1]))
    circuito.append(cirq.rz(z2).on(qubits[1]))
    circuito += agrupamiento_dos_qubits(qubits[0], qubits[1], agrupamiento)
    return circuito


# TODO: comprueba con un print(construir_circuito_sin_entrelazamiento(qubits)) que el CNOT inicial ya no aparece

In [ ]:
# TODO: monta el modelo de Keras con tfq.layers.PQC, igual que en la clase de teoría,
# pero usando construir_circuito_sin_entrelazamiento en vez de construir_circuito.
# Llama a la variable del modelo "modelo_sin_entrelazamiento" para no confundirla con el modelo original.

operador_lectura = [cirq.Z(qubits[1])]
entradas = tf.keras.Input(shape=(), dtype=tf.dtypes.string)
capa_cuantica = tfq.layers.PQC(
    # TODO: sustituye None por construir_circuito_sin_entrelazamiento(qubits)
    None, operador_lectura, repetitions=32,
    differentiator=tfq.differentiators.ParameterShift(),
    initializer=tf.keras.initializers.Zeros,
)(entradas)
modelo_sin_entrelazamiento = tf.keras.models.Model(inputs=entradas, outputs=capa_cuantica)
modelo_sin_entrelazamiento.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.01),
                                    loss=tf.keras.losses.hinge, metrics=[precision_hinge])

In [ ]:
# TODO: reentrena desde cero con las mismas 512 épocas y los mismos datos del XOR
historial_sin_entrelazamiento = None  # sustituye None por la llamada a .fit(...)

In [ ]:
# TODO: vuelve a dibujar la frontera de decisión, ahora con modelo_sin_entrelazamiento,
# barriendo el mismo plano [-0.1, 1.1] x [-0.1, 1.1] que en la clase de teoría.

mapa_colores = ListedColormap(["midnightblue", "maroon"])
N = 150
x1_malla, x2_malla = np.meshgrid(np.linspace(-0.1, 1.1, N), np.linspace(-0.1, 1.1, N))
puntos = np.c_[x1_malla.ravel(), x2_malla.ravel()]

# TODO: sustituye "modelo_sin_entrelazamiento" si has usado otro nombre de variable
predicciones = [None for p in puntos]  # TODO: calcula la predicción de cada punto
predicciones = np.asarray(predicciones).reshape(x1_malla.shape)

plt.pcolormesh(x1_malla, x2_malla, predicciones, cmap=mapa_colores)
plt.scatter([0, 1], [0, 1], color="blue", label="-1")
plt.scatter([0, 1], [1, 0], color="red", label="1")
plt.xlabel("x")
plt.ylabel("y")
plt.legend()
plt.show()

# TODO: compara historial.history["val_precision_hinge"][-1] (modelo original)
# con historial_sin_entrelazamiento.history["val_precision_hinge"][-1] (este modelo)

<details>
<summary>💡 Ver solución</summary>

```python
def construir_circuito_sin_entrelazamiento(qubits):
    """Igual que construir_circuito, pero sin el CNOT inicial que entrelaza los qubits de entrada."""
    x1, y1, z1, x2, y2, z2 = sympy.symbols("x1 y1 z1 x2 y2 z2")
    agrupamiento = sympy.symbols("agrupamiento0:6")
    circuito = cirq.Circuit()
    # Sin el CNOT inicial: los dos qubits ya no se entrelazan antes de las rotaciones
    circuito.append(cirq.rx(x1).on(qubits[0]))
    circuito.append(cirq.ry(y1).on(qubits[0]))
    circuito.append(cirq.rz(z1).on(qubits[0]))
    circuito.append(cirq.rx(x2).on(qubits[1]))
    circuito.append(cirq.ry(y2).on(qubits[1]))
    circuito.append(cirq.rz(z2).on(qubits[1]))
    circuito += agrupamiento_dos_qubits(qubits[0], qubits[1], agrupamiento)
    return circuito


operador_lectura = [cirq.Z(qubits[1])]
entradas = tf.keras.Input(shape=(), dtype=tf.dtypes.string)
capa_cuantica = tfq.layers.PQC(
    construir_circuito_sin_entrelazamiento(qubits), operador_lectura, repetitions=32,
    differentiator=tfq.differentiators.ParameterShift(),
    initializer=tf.keras.initializers.Zeros,
)(entradas)
modelo_sin_entrelazamiento = tf.keras.models.Model(inputs=entradas, outputs=capa_cuantica)
modelo_sin_entrelazamiento.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.01),
                                    loss=tf.keras.losses.hinge, metrics=[precision_hinge])

historial_sin_entrelazamiento = modelo_sin_entrelazamiento.fit(
    entrenamiento, etiquetas, epochs=512, batch_size=4,
    validation_data=(entrenamiento, etiquetas),
)

mapa_colores = ListedColormap(["midnightblue", "maroon"])
N = 150
x1_malla, x2_malla = np.meshgrid(np.linspace(-0.1, 1.1, N), np.linspace(-0.1, 1.1, N))
puntos = np.c_[x1_malla.ravel(), x2_malla.ravel()]
predicciones = [
    modelo_sin_entrelazamiento(
        tfq.convert_to_tensor([codificar_dato(qubits, cirq.Circuit(), p)])
    ).numpy().flatten()[0]
    for p in puntos
]
predicciones = np.asarray(predicciones).reshape(x1_malla.shape)

plt.pcolormesh(x1_malla, x2_malla, predicciones, cmap=mapa_colores)
plt.scatter([0, 1], [0, 1], color="blue", label="-1")
plt.scatter([0, 1], [1, 0], color="red", label="1")
plt.xlabel("x")
plt.ylabel("y")
plt.legend()
plt.show()

precision_original = historial.history["val_precision_hinge"][-1]
precision_sin_entrelazamiento = historial_sin_entrelazamiento.history["val_precision_hinge"][-1]
print("Precisión con CNOT inicial:", precision_original)
print("Precisión sin CNOT inicial:", precision_sin_entrelazamiento)
```

Sin el `CNOT` inicial, el circuito trata cada qubit de entrada por separado: la rotación de `qubits[0]` solo depende del primer bit de entrada, y la de `qubits[1]` solo del segundo. El agrupamiento final vuelve a mezclar los dos qubits, pero ya es demasiado tarde, porque la información de entrada nunca llegó a combinarse antes de esa mezcla. El resultado es un modelo que se comporta como dos clasificadores independientes, exactamente igual que un perceptrón clásico, y su precisión final se queda estancada alrededor del 50%: el circuito no distingue las cuatro combinaciones del XOR, tal y como predice la explicación teórica sobre la necesidad del entrelazamiento.

</details>